# 用 LLM Wiki 做一份教學 HTML 簡報：貝氏統計入門

本 notebook 分四步：**(1)** 以「先討論、後生成」建立貝氏統計的大型語言模型（LLM）Wiki；**(2)** 以 Wiki 為唯一知識來源，經過一個有檢查與重生機制的 graph 產生教學 HTML 簡報；**(3)** 說明 loop / graph 設計；**(4)** 比較兩次實際執行並反思。

所有函式都放在 `.py` 模組，notebook 只負責呼叫與呈現結果：

| 模組 | 內容 |
|---|---|
| `llm.py` | aisuite 包裝（JSON 解析失敗自動重試）、執行紀錄 `RunLog` |
| `wiki_tools.py` | 取得資料、ingest（先討論後生成）、lint、檢索、匯出 wiki.html |
| `slide_graph.py` | 簡報 graph：plan → compute_facts → retrieve → draft ⇄ check → deck_review → render |
| `report.py` | 流程圖、執行比較表、繳交說明草稿 |

## 0. 環境設定
在 Colab：第一格會請你上傳 `bayes_wiki_src.zip`（內含 .py 模組）；在本機（VS Code）：把 .py 檔與 notebook 放在同一資料夾。金鑰讀法同課堂 Demo07。

In [1]:
import os, zipfile
try:
    from google.colab import files, userdata
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB and not os.path.exists("slide_graph.py"):
    up = files.upload()                        # 選 bayes_wiki_src.zip
    zipfile.ZipFile(next(iter(up))).extractall(".")
%pip -q install aisuite groq openai markdown pandas scipy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
def get_key(name, env):
    return userdata.get(name) if IN_COLAB else os.environ[env]   # 本機請先設定環境變數

#【使用 Groq】llama-3.3-70b-versatile 已於 2026/8 下架
os.environ["GROQ_API_KEY"] = get_key("Groq", "GROQ_API_KEY")
PROVIDER, MODEL = "groq", "openai/gpt-oss-120b"

#【使用 OpenAI】
# os.environ["OPENAI_API_KEY"] = get_key("OpenAI", "OPENAI_API_KEY")
# PROVIDER, MODEL = "openai", "gpt-4o-mini"

In [3]:
from llm import LLM, RunLog
from wiki_tools import Wiki, DEFAULT_SOURCES, fetch_wikipedia, ingest, lint, export_html
from slide_graph import Config, run_graph, render_html, check_record
from report import show_graph, compare_runs, make_submission
from IPython.display import Markdown, display

log = RunLog()
llm = LLM(PROVIDER, MODEL, log=log)
wiki = Wiki("wiki")

## 1. 步驟一：建立 LLM Wiki

### 1.1 取得資料
預設 ingest 6 篇英文維基百科條目（作業要求至少 5 篇）。選這幾篇是因為它們剛好構成一條入門學習路徑：貝氏定理 → 基礎比率謬誤（最直觀的動機）→ 貝氏推論 → 先驗 → 共軛先驗 → 可信區間。原始資料存在 `wiki/raw/`，之後不再修改。

In [4]:
for s in DEFAULT_SOURCES:
    if not os.path.exists(f"wiki/raw/{s['name']}.txt"):    # 已抓過就不重抓
        text = fetch_wikipedia(s["title"])                 # 內建 429 限流重試
        wiki.save_raw(s["name"], text, meta=f"source: https://en.wikipedia.org/wiki/{s['title'].replace(' ', '_')}")
    print(f"{s['name']:<20} {len(wiki.read_raw(s['name'])):>6} 字元")

bayes_theorem          8054 字元
base_rate_fallacy      8057 字元
bayesian_inference     8058 字元
prior_probability      8057 字元
conjugate_prior        7770 字元
credible_interval      4917 字元


### 1.2 先討論、後生成（ingest）
每篇資料分兩個 LLM 呼叫：
1. **`ingest.discuss`**：只討論、不寫檔。LLM 讀新資料與目前的 `index.md`，列出重點、與既有頁面的衝突、建議新增／更新哪些頁，並反問我問題。討論結果存成 `wiki/ingest/<名稱>.discuss.json`。
2. **我的回饋**：讀完討論後回覆意見，原文記在下方 `NOTES`，透過 `notes=` 傳入。
3. **`ingest.generate`**：依討論與回饋寫出或合併頁面，再用規則檢查（有無 `# 標題`、`## 來源`、是否列出本篇資料），不過就帶著問題重生一次。

6 篇是逐篇互動完成的（看完一篇的討論、回覆後才進行下一篇）；每篇完成後存成 `wiki/ingest/<名稱>.json`，下面這格重新執行時直接讀取紀錄，不重打 API，也不會覆蓋當時的回饋。每次 ingest 都會更新 `index.md` 並在 `log.md` 留下回饋與寫入的頁面。

In [5]:
NOTES = {   # 我讀完每篇討論結果後的回覆（原文）
    "bayes_theorem": "照建議",
    "base_rate_fallacy": "1.好 2.不用",
    "bayesian_inference": "可以",
    "prior_probability": "好",
    "conjugate_prior": "不要",
    "credible_interval": "不用",
}
ingest_records = []
for s in DEFAULT_SOURCES:
    print(f"\n{'=' * 20} ingest：{s['name']} {'=' * 20}")
    ingest_records.append(ingest(llm, wiki, s["name"], wiki.read_raw(s["name"]), notes=NOTES[s["name"]]))
display(Markdown(wiki.index_text()))


==================== ingest：bayes_theorem ====================

── 討論結果 ──
摘要： 這份資料說明貝氏定理（Bayes' theorem）的概念、歷史背景、數學式與醫學診斷等範例，並提到其在貝氏推論中的角色。
  • 貝氏定理提供從條件機率反推原因機率的公式。
  • 公式：$P(A|B)=\frac{P(B|A)P(A)}{P(B)}$，其中 $P(A)$ 為先驗機率，$P(A|B)$ 為後驗機率。
  • 若事件 $A_i$ 為互斥且完整，則可使用全機率定理擴展：$P(A_i|B)=\frac{P(A_i)P(B|A_i)}{\sum_j P(A_j)P(B|A_j)}$。
  • 歷史上由 Thomas Bayes 於 18 世紀提出，後由 Pierre‑Simon Laplace 獨立發展並擴充。
  • 貝氏定理是貝氏推論（Bayesian inference）的核心，用於把似然函數（likelihood）與先驗結合得到後驗分佈（posterior distribution）。
  • 醫學診斷是常見範例：利用疾病盛行率與檢驗靈敏度/特異度計算陽性結果的後驗機率。
  → create [[bayes-theorem]]：說明貝氏定理的定義、推導與基本範例，是貝氏統計入門的基礎頁面。
  → create [[bayesian-inference]]：資料中提到貝氏推論是貝氏定理的主要應用，需要獨立說明概念與流程。
  → create [[medical-diagnosis-example]]：提供醫學診斷的具體計算示例，幫助初學者理解公式的實際運用。
  ？ 您希望在範例中使用哪種情境（例如醫學診斷、藥物測試或其他）作為主要示範？
  ？ 是否需要在頁面中加入貝氏定理的圖示或直觀說明，以協助初學者更快掌握概念？

我的回饋：照建議
→ 寫入 ['bayes-theorem', 'bayesian-inference', 'medical-diagnosis-example']（已完成，讀取紀錄）

==================== ingest：base_rate_fallacy ====================

── 討論結果 ──
摘要： 這份資料說明基礎率

# 貝氏統計入門 Wiki 目錄

| 頁面 | 標題 | 摘要 |
|---|---|---|
| [[base-rate-fallacy]] | 基礎率謬誤 | 基礎率謬誤（base rate fallacy）是指在判斷時忽略整體發生率（先驗機率），只看個別案例的資訊而產生的錯誤推論。 |
| [[bayes-theorem]] | Bayes' Theorem | 貝氏定理（Bayes' theorem）提供從條件機率反推原因機率的公式。 |
| [[bayesian-inference]] | Bayesian Inference | 貝氏推論（Bayesian inference）利用貝氏定理（Bayes' theorem）將先驗機率（prior probability）與觀測資料的概似函數（likelihood）結合，得到後驗分佈（posterior distribu |
| [[bayesian-updating]] | Bayesian Updating | 貝氏更新（Bayesian updating）是指在觀測到新證據 $E$ 後，依據貝氏定理將先驗機率 $P(H)$ 更新為後驗機率 $P(H\mid E)$ 的過程，並可在後續觀測到更多證據時重複此步驟，形成動態的信念修正機制。 |
| [[beta-binomial-example]] | 二項‑貝塔範例（Beta‑Binomial Example） | 說明二項分佈的概似與 Beta 先驗的共軛關係，以及超參數的更新方式。 |
| [[conjugate-prior]] | 共軛先驗（Conjugate Prior） | 此頁說明共軛先驗的概念、超參數的意義與常見範例，並說明其在貝氏推論中的便利性。 |
| [[credible-interval]] | 可信區間（credible interval） | 用於描述參數在後驗分佈（posterior distribution）中落於某區間的機率。 |
| [[false-positive-paradox]] | 假陽性悖論 | 假陽性悖論（false positive paradox）說明在低發生率族群中，即使測試的準確度極高，陽性結果仍可能大多是偽陽性。 |
| [[gamma-poisson-example]] | Gamma‑Poisson 範例（Gamma‑Poisson Example） | 展示 Gamma 分佈作為泊松分佈（Poisson）之共軛先驗，並說明後驗預測為負二項分佈的計算步驟。 |
| [[informative-prior]] | 資訊性先驗（Informative Prior） | 說明資訊性先驗的概念、強弱分類與典型應用情境。 |
| [[medical-diagnosis-example]] | Medical Diagnosis Example | 本範例說明如何利用貝氏定理（[[bayes-theorem]]）計算患者在檢驗陽性時實際患病的機率。 |
| [[prior-probability]] | 先驗機率（Prior Probability） | 說明先驗機率與先驗分佈的定義、類型與建構方式。 |
| [[uninformative-prior]] | 無資訊性先驗（Uninformative Prior） | 說明無資訊性先驗的定義、常見形式與哲學背景。 |


### 1.3 Lint：整份 Wiki 的健康檢查
規則檢查斷掉的 `[[連結]]` 與沒有被連到的孤兒頁；LLM 檢查頁面間矛盾與統計觀念錯誤，並自動修正。

In [6]:
lint_report = lint(llm, wiki)
export_html(wiki, "wiki.html")
lint_report["issues"]

（已完成，讀取紀錄）斷連結 0 個；孤兒頁 ['beta-binomial-example', 'gamma-poisson-example', 'conjugate-prior', 'credible-interval']；LLM 發現 0 個問題並已修正


[]

Wiki 成果：`wiki/pages/*.md`、`wiki/index.md`、`wiki/log.md`，以及單頁瀏覽版 `wiki.html`。

In [7]:
display(Markdown(open("wiki/log.md", encoding="utf-8").read()[-3000:]))

# Wiki 變更紀錄

## [2026-11-13 14:20] ingest | bayes_theorem
- 使用者回饋：照建議
- 寫入頁面：bayes-theorem, bayesian-inference, medical-diagnosis-example
- 說明：新增 bayes-theorem、bayesian-inference、medical-diagnosis-example 三頁，說明定理、推論流程與醫學診斷範例。

## [2026-11-13 14:23] ingest | base_rate_fallacy
- 使用者回饋：1.好 2.不用
- 寫入頁面：base-rate-fallacy, false-positive-paradox
- 說明：新增 base-rate-fallacy 與 false-positive-paradox 兩頁，說明概念、例子與貝氏推論關聯

## [2026-11-13 14:24] ingest | base_rate_fallacy
- 使用者回饋：1.好 2.不用
- 寫入頁面：base-rate-fallacy, false-positive-paradox
- 說明：在 base-rate-fallacy 頁面加入醫學診斷連結，保持原有內容；更新 false-positive-paradox 內容保持一致。

## [2026-11-13 14:25] ingest | bayesian_inference
- 使用者回饋：可以
- 寫入頁面：bayesian-inference, bayesian-updating
- 說明：在 bayesian-inference 頁面加入實務範例與非貝氏更新差異說明，並新增 bayesian-updating 子頁說明單次與多次更新流程

## [2026-11-13 14:27] 人工修正 | false-positive-paradox
- 原因：ingest base_rate_fallacy 時使用者同意「新頁面加入 [[medical-diagnosis-example]] 連結」，重新生成後只有 base-rate-fallacy 加上
- 修改：在「相關頁面」補上 [[medical-diagnosis-example]]

## [2026-11-13 14:27] ingest | prior_probability
- 使用者回饋：好
- 寫入頁面：prior-probability, informative-prior, uninformative-prior
- 說明：新增先驗機率、資訊性先驗、無資訊性先驗三頁，並加入數學範例與應用說明

## [2026-11-13 14:28] ingest | conjugate_prior
- 使用者回饋：不要
- 寫入頁面：conjugate-prior, beta-binomial-example, gamma-poisson-example
- 說明：新增共軛先驗及兩個具體範例頁面

## [2026-11-13 14:29] ingest | credible_interval
- 使用者回饋：不用
- 寫入頁面：credible-interval
- 說明：新增 credible-interval 頁面，說明定義、類型、與信賴區間的比較及計算方法。

## [2026-11-13 14:38] lint
- 斷連結：[]
- 孤兒頁：['beta-binomial-example', 'gamma-poisson-example', 'conjugate-prior', 'credible-interval']
- LLM 檢查問題：

## [2026-11-14 14:34] 人工修正 | 公式與用語
- false-positive-paradox：`999 × 0.05 \approx 49.95` 不在 $…$ 內，wiki.html 顯示原始 LaTeX（lint 未抓到），補上 $…$
- 用語統一為台灣習慣（似然→概似、分布→分佈、信息→資訊），影響 10 頁：base-rate-fallacy, bayes-theorem, bayesian-inference, bayesian-updating, beta-binomial-example, conjugate-prior, credible-interval, gamma-poisson-example, prior-probability, uninformative-prior


## 2. 步驟二：用 Wiki 產生教學 HTML 簡報

### 2.1 Graph 設計

In [8]:
show_graph()

| 節點 | 做什麼 | 檢查 | 不過時 |
|---|---|---|---|
| `plan` | 讀 `index.md` 規劃 6–7 張大綱，每張指定依據頁面與範例 | `check_plan`：頁面是否存在、張數、兩個範例都有用到 | 帶問題重新規劃（≤3 次） |
| `compute_facts` | Python 算出疾病篩檢與擲硬幣範例的所有數字 | 程式計算，不需檢查 | — |
| `retrieve` | 取大綱指定頁面＋關鍵字檢索補 2 頁 | — | — |
| `draft` | 只根據取回的 Wiki 內容寫單張投影片（JSON） | 見下列三層檢查 | 帶問題重生（≤3 版），仍不過標記 ⚑ |
| `check.rule` | 格式：2–5 點、每點 ≤60 字、sources 存在於 Wiki | 規則 | 回 `draft` |
| `check.numeric` | 用 Python 重算 `calc` 欄位與內文百分比 | 規則 | 回 `draft` |
| `check.judge` | LLM 審稿：每句是否有 Wiki 依據、初學者是否看得懂 | LLM | 回 `draft` |
| `deck_review` | 看整份簡報的順序、重複、缺漏 | LLM | 指定的投影片再跑一次 loop |
| `render_html` | 輸出簡報，每頁列出依據的 Wiki 頁面並可點回 wiki.html | — | — |

### 2.2 第一次執行（v1）：數字交給 LLM 自己算
v1 把範例參數交給 LLM，請它自己計算後驗機率，數字錯誤只能等到 `check.numeric` 才發現。

> 本次使用 Groq 免費額度（每日約 20 萬 token）。為了在一天內完成 v1、v2 兩次執行，兩版都設定為 6–7 張內容頁、retrieve 只補 1 頁（預設為 8–11 張、補 2 頁）；lint 與 graph 另存檢查點（`wiki/lint.json`、`ckpt_v*.json`），額度用完時可隔天接續。

In [9]:
SMALL = dict(n_slides=(6, 7), k_extra=1)   # 配合免費額度：6–7 張、每張少帶 1 頁 Wiki（v1、v2 相同）
cfg_v1 = Config(precompute_numbers=False, **SMALL)
llm.log = log_v1 = RunLog()
res_v1 = run_graph(llm, wiki, cfg_v1, ckpt="ckpt_v1.json")   # 每日額度用完時隔天重跑，從檢查點接續
calls_v1 = res_v1["llm_calls"]
render_html(res_v1, wiki, cfg_v1, "slides_v1.html")
log_v1.save(".", "run_log_v1")
display(Markdown(check_record(res_v1)))

# 投影片檢查紀錄

| # | 標題 | 生成次數 | 結果 | 被擋下的原因 |
|---|---|---|---|---|
| 1 | 什麼是先驗機率與後驗機率？ | 5 | 通過 | 第1版：初學者不易懂，未定義術語：['似然', '證據的邊際機率', '遞迴更新']; 審稿建議：1. 在第一張投影片加入「似然」的簡要說明，例如：'似然是給定參數時觀測到資料的機率'。 2. 解釋「證據的邊際機率」為何需要，並給出計算方式的直觀描述。 3. 將「遞迴更新」改寫為「逐次更新」或在備註中說明其含義，避免使用未解釋的術語。 4. 若時間允許，可在例子中加入硬幣投擲或醫學檢測的具體數值示範，幫助學生把抽象概念具體化。 / 第2版：後驗平均寫成 0.75，正確是 0.6429（Beta(9,5)） / 第3版：後驗平均寫成 0.667，正確是 0.6429（Beta(9,5)） / 整體審稿：在首次提到「似然」前先加入簡短定義，例如「似然是給定參數時觀測資料出現的機率」，並說明它與先驗的關係，避免概念先行。 / 第1版：初學者不易懂，未定義術語：['Beta(α,β) 分佈']; … |
| 2 | 貝氏定理的推導：從全機率到後驗機率 | 4 | 通過 | 第1版：初學者不易懂，未定義術語：['靈敏度', '偽陽性率', '全機率定理']; 審稿建議：整體內容與 Wiki 資料相符，統計概念正確。建議在投影片中加入簡短定義： - 靈敏度 (Sensitivity) = P(陽性／有病) - 偽陽性率 (False Positive Rate) = P(陽性／無病) - 全機率定理的說明與公式，讓未接觸過的學生能立即理解。可在例子後補充計算步驟，展示分母 P(B) 的展開，以避免概念跳躍。 / 第2版：缺乏 Wiki 依據或觀念錯誤：['似然 $P(B／A)$：在 A 成立時觀測到 B 的機率，稱為靈敏度。', '偽陽性率 $P(B／\\neg A)$：無病時仍出現陽性的機率。']; 初學者不易懂，未定義術語：['靈敏度', '偽陽性率']; 審稿建議：1. 將「靈敏度」改為「似然」或在說明中註明：在醫學檢驗中，似然常被稱為靈敏度。2. 為「偽… |
| 3 | 基礎率謬誤：忽略先驗的危險 | 6 | ⚑ 待人工確認 | 第1版：初學者不易懂，未定義術語：['靈敏度', '事後機率', '似然', 'posterior']; 審稿建議：內容大體正確且都有 Wiki 依據。建議在投影片中加入以下說明： 1. 定義「靈敏度」(Sensitivity) 為 P(+／D)； 2. 解釋「事後機率」即貝氏定理算出的 P(D／+)； 3. 簡要說明「似然」是檢驗結果在不同假設下的機率； 4. 在公式下方或註解中說明「posterior」指的就是事後機率。加入這些定義後，初學者即可完全理解。 / 第2版：缺乏 Wiki 依據或觀念錯誤：['靈敏度（Sensitivity）＝P(+／D)，即疾病時測出陽性的機率。']; 初學者不易懂，未定義術語：['靈敏度', '偽陽性率']; 審稿建議：1. 為了確保每個重點都有來源，請在投影片中加入對「靈敏度」的說明，並引用相關的 Wiki 條目（例如在 Bayes 定理的說明中提到 … |
| 4 | 貝氏推論：結合先驗與似然 | 1 | 通過 | — |
| 5 | 共軛先驗與 Beta‑Binomial 範例 | 1 | 通過 | — |
| 6 | 可信區間：從點估計到區間估計 | 3 | ⚑ 待人工確認 | 第1版：初學者不易懂，未定義術語：['後驗分佈', '最高密度區間', 'MCMC', '解析法', '等尾可信區間', '最小可信區間']; 審稿建議：建議在投影片中加入簡短定義： - 後驗分佈：結合先驗與資料後得到的參數分佈。 - 最高密度區間（最小可信區間）：密度最高的區間，包含指定機率。 - 等尾可信區間：取兩側分位數形成的區間。 - 解析法：利用已知分布的公式直接計算分位數或密度等高線。 - MCMC：一種透過隨機抽樣近似後驗分佈的方法，可簡要說明如「馬可夫鏈蒙特卡羅」。 此外，可在「參數是隨機的」說明後，加上一兩句關於先驗的概念，避免聽眾誤以為參數本身隨機。 / 第2版：初學者不易懂，未定義術語：['MCMC（馬可夫鏈蒙特卡羅）', '最高密度區間（smallest credible interval）', '等尾可信區間（median credible interval）',… |
| 7 | 實務應用：疾病篩檢的貝氏更新 | 2 | 通過 | 第1版：缺乏 Wiki 依據或觀念錯誤：['「檢驗靈敏度高達 90%」的數值未在 Wiki 內容中出現，Wiki 中的例子使用 0.99 或 0.99 的靈敏度。', '計算得到的後驗機率 0.1538 與 Wiki 中提供的示例數值不符，且 Wiki 未給出對應的先驗機率 1% 與偽陽性率 5% 的組合，因此此具體結果缺乏依據。']; 審稿建議：建議將數值調整為與 Wiki 例子一致，或在投影片說明中註明所使用的參數來源。例如，可使用 Wiki 中的 0.001、0.99、0.02 參數得到約 4.7% 的後驗機率，或在說明中加入對 1% 先驗與 5% 偽陽性率的來源說明。其他概念與術語均已在 Wiki 中定義，適合基礎機率背景的學生。 |

**整體審稿（deck_review）**：整體結構清晰，概念逐步深化，但部分關鍵術語（如似然、先驗）出現順序不佳，導致初學者可能感到困惑。調整上述三張投影片的內容與順序，可提升概念的前置鋪陳與避免重複，使學習流程更流暢。
需修改：[1, 2, 3]

### 2.3 第二次執行（v2）：把數字檢查往前移
依 v1 的紀錄調整 graph：在 `draft` 之前加入 `compute_facts`，由 Python 先算好範例數字並當作「已驗證事實」交給 `draft`，`check.numeric` 保留作為最後防線。

In [10]:
cfg_v2 = Config(precompute_numbers=True, **SMALL)
llm.log = log_v2 = RunLog()
res_v2 = run_graph(llm, wiki, cfg_v2, ckpt="ckpt_v2.json")
calls_v2 = res_v2["llm_calls"]
render_html(res_v2, wiki, cfg_v2, "slides.html")
log_v2.save(".", "run_log_v2")
display(Markdown(check_record(res_v2)))

# 投影片檢查紀錄

| # | 標題 | 生成次數 | 結果 | 被擋下的原因 |
|---|---|---|---|---|
| 1 | 什麼是先驗機率與後驗機率？ | 3 | 通過 | 第1版：初學者不易懂，未定義術語：['似然', '證據', '遞迴更新', '動態的信念修正機制']; 審稿建議：內容大體正確且都有 Wiki 依據。為了讓未學過貝氏統計的學生易於理解，建議在投影片中加入以下說明： 1. 簡短定義「似然」：在給定假說 H 下，觀測到資料 E 的機率。 2. 解釋「證據」\(P(E)\)：所有可能假說的加權總和，使後驗機率正規化。 3. 用更口語化的說法說明「遞迴更新」：每取得一次新觀測，就把上一次的後驗當作新的先驗，再套用貝氏定理。 4. 「動態的信念修正機制」可改寫為「隨著新資料不斷加入，我們的信念會持續更新」。 加入上述說明後，術語即不會未定義，初學者即可跟上。 / 第2版：缺乏 Wiki 依據或觀念錯誤：['「似然是指在假說 H 成立時，觀測到資料 E 的機率。」——雖然概念在 Wiki 中提及，但未明確以此文字說明。', '「證據 P(E) 是所有… |
| 2 | 貝氏定理：從條件機率到原因機率 | 6 | ⚑ 待人工確認 | 第1版：重點過長：['貝氏定理把已知條件下的機率 $…', '公式為 $P(A／B)=\\fr…', '若有多個互斥且完整的假設 $A…'] / 第2版：初學者不易懂，未定義術語：['正規化']; 審稿建議：建議在投影片或備註中加入「正規化」的簡短說明，例如：在多個假設下，分母的求和把所有可能的情況加起來，使後驗機率的總和為 1。此外，可補充一個簡單的例子（如醫學檢驗或硬幣投擲）說明先驗、似然與後驗的關係，並明確指出 $P(B)$ 需要透過全機率定理計算，而不是固定常數。 / 第3版：重點過長：['在多個假設 $A_1\\dots…'] / 整體審稿：在首次提到「全機率定理」前，先加入一張簡短說明全機率定理的投影片，或在本頁的第一個子彈加入「全機率定理：若 {A₁…A_k} 為互斥且完備的假設，則 P(B)=∑ₖ P(A_k)P(B／A_k)」的簡要說明，避免概念先行。 / 第1版：重點過… |
| 3 | 基礎率謬誤與假陽性悖論 | 3 | ⚑ 待人工確認 | 第1版：重點過長：['在 disease_test …'] / 第2版：缺乏 Wiki 依據或觀念錯誤：['disease_test：先驗 1%、靈敏度 90%、偽陽性 5% 時，後驗約 15.4%。']; 審稿建議：將 disease_test 的數值說明改為與 Wiki 中的範例一致，或在投影片註明此為自行假設的示例並提供計算過程。其餘內容皆有對應的 Wiki 依據，且術語已在來源中解釋，適合基礎機率背景的學生。 / 第3版：這張應該有 disease_test 範例的 calc 欄位 |
| 4 | 貝氏推論的流程 | 1 | 通過 | — |
| 5 | 二項與 Beta‑先驗範例 | 5 | 通過 | 第1版：重點過長：['使用 Beta‑Binomia…'] / 第2版：初學者不易懂，未定義術語：['MLE']; 審稿建議：整體內容與 Wiki 資料相符，統計概念正確。建議在投影片中加入對 'MLE'（最大概似估計）的簡短說明，或改用「樣本比例」等更易懂的詞彙，以免未接觸過此概念的學生感到困惑。 / 第3版：初學者不易懂，未定義術語：['95% 信賴區間', '點估計']; 審稿建議：整體內容與 Wiki 資料相符，統計概念正確。建議在投影片中簡要說明「95% 信賴區間」與「點估計」的意義，或改用「後驗分布的可信區間」等較易理解的說法，避免未定義術語影響初學者理解。 / 整體審稿：在介紹 Beta‑Binomial 前，先加入一張說明「二項分布」與「成功/失敗」概念的投影片，確保學生已熟悉二項模型的基本概念，避免直接使用 s、f 造成理解斷層。 / 第1版：重點過長：['Beta‑Bi… |
| 6 | 可信區間與結果解讀 | 6 | ⚑ 待人工確認 | 第1版：初學者不易懂，未定義術語：['後驗分佈', '密度', '模式', '中位數', '解析法', 'MCMC']; 審稿建議：建議在投影片中加入簡短定義： - 後驗分佈：在觀測資料後，參數的機率分佈。 - 密度：指機率密度函數的高度，表示參數取某值的相對可能性。 - 模式 (mode)：後驗分佈中機率密度最高的點。 - 中位數：使分佈左右各有 50% 資料的值。 - 解析法：利用已知的數學公式直接求區間，例如正態或貝塔分佈。 - MCMC：馬可夫鏈蒙特卡羅，用電腦模擬抽樣來近似後驗分佈。 此外，可在備註中加入「可信區間」的頻率解釋差異示意圖，幫助未接觸貝氏統計的同學快速理解。 / 第2版：初學者不易懂，未定義術語：['SCI', 'MCI', 'MCMC']; 審稿建議：1. 在首次出現 SCI、MCI、MCMC 時加入簡短說明，例如： - SCI（Smallest Credible… |
| 7 | 醫學診斷實務：結合所有概念 | 2 | 通過 | 第1版：缺乏 Wiki 依據或觀念錯誤：['先驗機率（先驗）代表疾病在一般族群的發生率，這裡是 1%。', '靈敏度是測試在真患病者中呈陽性的機率，給定為 90%。', '偽陽性率是測試在未患病者中誤判為陽性的機率，這裡是 5%。', '在 10,000 人的族群中，根據給定參數，真陽性 90 人、偽陽性 495 人，最終陽性者中實際患病機率為 15.4%。']; 審稿建議：1. 將所有數值改為 Wiki 中提供的範例或明確說明是自行假設的情境，並在投影片註記來源。 2. 若要保留 1% 的先驗機率，需在 Wiki 內加入相應的說明或引用其他可靠來源。 3. 公式本身正確，可保留。 4. 建議在投影片加入「先驗機率 (prevalence)」與「特異度」的簡短定義，以避免初學者混淆。 |

**整體審稿（deck_review）**：整體結構清晰，概念大多循序漸進，但有少數關鍵術語（全機率定理、二項分布、SCI/MCI）在使用前未充分鋪陳，且部分進階內容對初學者可能造成負擔。調整上述三張投影片即可提升易懂度與連貫性。
需修改：[2, 6, 5]

### 2.4 兩次執行比較
各節點擋下（失敗）的次數與 LLM 呼叫數：

In [11]:
compare_runs({"v1": log_v1, "v2": log_v2}, calls={"v1": calls_v1, "v2": calls_v2})

,v1 失敗,v1 總次數,v2 失敗,v2 總次數
check.judge,13,19,13,18
check.numeric,2,24,0,26
check.rule,2,24,8,26
deck_review,1,1,1,1
check_plan,0,1,0,1
retrieve,0,11,0,10
draft,0,6,0,5
（LLM 呼叫數）,0,46,0,46


## 3. 步驟三：loop / graph 設計說明

**流程**：取得資料 → ingest（先討論 → 人工回饋 → 後生成 → 規則檢查）→ lint → plan ⇄ check_plan → compute_facts → 每張投影片 retrieve → draft ⇄ check → deck_review → render。

**哪些步驟會重複執行？**
- ingest 對每篇資料各跑一次，後面的資料會讀到前面建立的頁面，因此可以合併、而不是重複建頁。
- retrieve → draft → check 對每張投影片各跑一次；draft ⇄ check 是內層 loop，最多 3 版。
- plan ⇄ check_plan 最多 3 次；deck_review 指出的投影片會再跑一次內層 loop。

**哪些地方需要檢查或重新生成？**
- ingest 後：頁面格式與來源（避免無法追溯的內容進入 Wiki）。
- lint：頁面間矛盾與觀念錯誤（Wiki 錯了，簡報一定錯）。
- check_plan：大綱引用的頁面必須存在，否則後面每一張都會取不到內容。
- check_slide：格式 → 數字 → LLM 審稿，由便宜到昂貴。
- deck_review：單張都通過不代表整份順暢，最後看整體順序與重複。

**為什麼這樣設計？**
1. **錯誤越早擋越便宜**：Wiki 層的錯誤會擴散到所有引用它的投影片，所以 Wiki 的檢查放在簡報生成之前。
2. **便宜的檢查先做**：規則與數字檢查不需呼叫 LLM，沒過就直接重生，不浪費 LLM 審稿的額度。
3. **不信任 LLM 的算術**：數字由 Python 計算，LLM 只負責解釋。
4. **可追溯**：每張投影片的 `sources` 必須是 Wiki 頁面，HTML 頁尾連回 wiki.html，Wiki 與簡報內容可以一一對照。
5. **有上限的 loop**：每個 loop 都設最大次數，超過就標記 ⚑ 交給人工確認，避免無限重試。

## 4. 步驟四：反思
依 2.2–2.4 兩次實際執行的紀錄（`run_log_v1.md`、`run_log_v2.md`）整理。

**哪一個節點最常出錯？**
`check.judge`（LLM 審稿）在兩次執行中都是失敗最多的節點：v1 審稿 19 次有 13 次不通過，v2 審稿 18 次有 13 次不通過。v1 的 13 次中有 11 次與「初學者不易懂、術語未定義」有關，而這項要求常和「每句都要有 Wiki 依據」互相拉扯。例如 v1 第 2 張第 1 版被判定「靈敏度、偽陽性率、全機率定理未定義」，第 2 版補上定義後，又被判定「『似然 $P(B|A)$ …稱為靈敏度』在 Wiki 中沒有依據」。v1 另一類錯誤是數字：第 1 張由 LLM 自己計算 Beta(9,5) 的後驗平均，第 2、3 版分別寫成 0.75 和 0.667（正確為 0.6429），兩次都被 `check.numeric` 擋下。

**發現問題後，把檢查往前移到哪一個步驟？**
- **數字**：從 draft 之後的 `check.numeric` 往前移到 draft 之前的 `compute_facts`，由 Python 先算好範例數字，當作已驗證事實交給 draft；`check.numeric` 保留為最後防線。v2 的數字錯誤由 2 次降為 0 次。
- **ingest 回饋**：第 2 篇我回覆「1.好」（同意加入連結），生成時卻被忽略，而只檢查格式的 `check_pages` 仍判定通過。之後把「LLM 提出的問題（編號）與我的回饋必須落實」直接寫進 `ingest.generate` 的指令，在生成當下就約束，而不是事後才發現。
- **JSON 與 token**：LaTeX 反斜線未跳脫造成的 JSON 解析失敗，改為先在本地修復再解析，不必把錯誤丟回模型重試；lint 一次送出整份 Wiki 會超過每分鐘 token 上限，改為送出前先分批。

**有沒有因此修改 workflow / graph？為什麼？**
有。v2 在 draft 之前加入 `compute_facts` 節點，因為 v1 顯示 LLM 的算術不可靠，而且錯誤要到 draft 之後才發現，每次都得多生成一輪。但結果顯示只把一個檢查往前移還不夠：v2 的 LLM 呼叫數仍是 46 次（與 v1 相同），標記 ⚑ 的投影片也從 2 張變成 3 張。原因是下游的檢查沒有一起調整：數字放進重點句後，「重點過長」從 2 次增加到 8 次；`check.judge` 看不到 Python 算好的事實，反而把第 3 張的「後驗約 15.4%」判為缺乏 Wiki 依據。下一步應把 `compute_facts` 的結果一併交給 `check.judge` 作為依據，並把術語檢查往前移到 `plan`：規劃大綱時就指定每張投影片要先定義哪些術語，避免 `check.judge` 到最後才一再擋下「術語未定義」。

## 5. 匯出繳交檔案

In [12]:
make_submission(".", wiki, {"v1": res_v1, "v2": res_v2}, {"v1": log_v1, "v2": log_v2}, cfg_v2, ingest_records,
                site="https://fylel.github.io/bayes-llm-wiki/", repo="https://github.com/fylel/bayes-llm-wiki")
import glob
with zipfile.ZipFile("bayes_wiki_output.zip", "w", zipfile.ZIP_DEFLATED) as z:   # Windows 沒有 zip 指令，改用 zipfile
    for f in [*glob.glob("wiki/**/*.*", recursive=True), "wiki.html", "slides.html", "slides_v1.html",
              *glob.glob("run_log_v*"), "繳交說明.md", "reflection.md", *glob.glob("*.py")]:
        z.write(f)
if IN_COLAB:
    files.download("bayes_wiki_output.zip")

**線上連結**：[教學簡報 slides.html](https://fylel.github.io/bayes-llm-wiki/slides.html)、[LLM Wiki wiki.html](https://fylel.github.io/bayes-llm-wiki/wiki.html)、[GitHub repo](https://github.com/fylel/bayes-llm-wiki)（GitHub Pages；`slides.html` 與 `wiki.html` 放在同一層，頁尾的 Wiki 連結才會正常）。